# Práctica — Clase 4. Programación dinámica

Esta práctica te toca a ti. Trabajas de forma autónoma sobre el mismo `GridWorld` que usaste en clase, y programas desde cero las dos piezas centrales de la **programación dinámica**: iteración de políticas e iteración de valores.

**Cómo trabajar con este notebook:**
1. Ejecuta primero la sección "Preparación", que reconstruye el entorno y las funciones de la clase anterior
2. En cada ejercicio, completa las líneas marcadas con `# TODO`
3. Ejecuta tu solución y compara el resultado con lo que se describe en el enunciado
4. Solo después, despliega la solución oculta para comprobar tu respuesta

> **No mires la solución antes de intentarlo.** El valor de este ejercicio está en pelearte un poco con la mejora de políticas y con la ecuación de optimalidad de Bellman, no en leer la respuesta.

## Preparación: recupera el entorno de la clase

Estas celdas reproducen tal cual el `GridWorld`, `politica_aleatoria`, `evaluacion_iterativa_politica` e `imprimir_politica` que ya conoces de las dos clases anteriores. La única novedad es un parámetro nuevo y opcional en `GridWorld`: `obstaculos`, una lista de casillas que el agente no puede atravesar —si intenta entrar en una, se queda en su sitio, igual que si chocara contra el borde del tablero—. Por defecto vale `None` y el GridWorld se comporta exactamente igual que en clase.

Los ejercicios de hoy reutilizan estos mismos nombres, así que no los cambies.

In [ ]:
import numpy as np

ACCIONES = ["arriba", "abajo", "izquierda", "derecha"]

DESPLAZAMIENTOS = {
    "arriba": (-1, 0),
    "abajo": (1, 0),
    "izquierda": (0, -1),
    "derecha": (0, 1),
}

FLECHAS = {"arriba": "^", "abajo": "v", "izquierda": "<", "derecha": ">"}

In [ ]:
class GridWorld:
    """
    Un mundo-cuadrícula de n_filas x n_columnas con una casilla terminal (la meta)
    y, opcionalmente, casillas-obstáculo.

    - Los estados son tuplas (fila, columna); los obstáculos no son estados
      habitables, así que no aparecen en mundo.estados.
    - Las transiciones son deterministas: moverse contra un muro, un obstáculo
      o el borde del tablero deja al agente en la misma casilla.
    - La recompensa es 'recompensa_paso' en cada movimiento (por defecto -1),
      salvo en la casilla terminal, donde el episodio ya ha terminado.
    """

    def __init__(self, n_filas=4, n_columnas=4, estados_terminales=None,
                 recompensa_paso=-1.0, obstaculos=None):
        self.n_filas = n_filas
        self.n_columnas = n_columnas
        self.recompensa_paso = recompensa_paso

        if estados_terminales is None:
            estados_terminales = [(n_filas - 1, n_columnas - 1)]
        self.estados_terminales = set(estados_terminales)
        self.obstaculos = set(obstaculos) if obstaculos else set()

        # Los obstáculos no son casillas habitables: el agente nunca "está" en ellas
        self.estados = [
            (f, c) for f in range(n_filas) for c in range(n_columnas)
            if (f, c) not in self.obstaculos
        ]
        self.acciones = ACCIONES

    def es_terminal(self, estado):
        return estado in self.estados_terminales

    def estado_siguiente(self, estado, accion):
        """Devuelve la casilla resultante de aplicar 'accion' desde 'estado'.
        Si el movimiento choca con el borde del tablero o con un obstáculo,
        el agente se queda quieto."""
        if self.es_terminal(estado):
            return estado

        df, dc = DESPLAZAMIENTOS[accion]
        fila, columna = estado
        nueva_fila = max(0, min(self.n_filas - 1, fila + df))
        nueva_columna = max(0, min(self.n_columnas - 1, columna + dc))
        nuevo_estado = (nueva_fila, nueva_columna)

        if nuevo_estado in self.obstaculos:
            return estado  # el obstáculo actúa como un muro
        return nuevo_estado

    def recompensa(self, estado, accion, estado_siguiente):
        if self.es_terminal(estado):
            return 0.0
        return self.recompensa_paso


def politica_aleatoria(mundo):
    """Construye una política estocástica que, en cada estado, elige
    cualquiera de las cuatro acciones con la misma probabilidad (0.25)."""
    probabilidad_uniforme = 1.0 / len(mundo.acciones)
    return {
        estado: {accion: probabilidad_uniforme for accion in mundo.acciones}
        for estado in mundo.estados
    }


def evaluacion_iterativa_politica(mundo, politica, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Aplica la ecuación de Bellman de expectación barrido tras barrido,
    hasta que Vpi deja de cambiar. Devuelve V y el número de barridos usados."""
    V = {estado: 0.0 for estado in mundo.estados}
    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)
        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue
            valor_esperado = 0.0
            for accion in mundo.acciones:
                prob_accion = politica[estado][accion]
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                valor_esperado += prob_accion * (r + gamma * V[estado_sig])
            V_nuevo[estado] = valor_esperado
            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))
        V = V_nuevo
        if delta_maximo < umbral:
            break
    return V, iteracion


def imprimir_politica(mundo, politica):
    """Muestra la política como una cuadrícula de flechas. Las casillas
    obstáculo se marcan con '#' y la meta con 'G'."""
    for f in range(mundo.n_filas):
        fila_textos = []
        for c in range(mundo.n_columnas):
            estado = (f, c)
            if estado in mundo.obstaculos:
                fila_textos.append(" # ")
            elif mundo.es_terminal(estado):
                fila_textos.append(" G ")
            else:
                mejor_accion = max(politica[estado], key=politica[estado].get)
                fila_textos.append(f" {FLECHAS[mejor_accion]} ")
        print("|".join(fila_textos))

### Línea base: el GridWorld original

Ejecuta la celda siguiente para tener a mano la misma referencia que viste en clase: el GridWorld 4×4 sin obstáculos, con la meta en la esquina inferior derecha. Vas a usar `mundo` y `gamma` en el ejercicio 1, así que no los sobrescribas.

In [ ]:
mundo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0)
gamma = 0.9

politica_inicial = politica_aleatoria(mundo)
V_inicial, barridos_iniciales = evaluacion_iterativa_politica(mundo, politica_inicial, gamma=gamma)

print(f"Vπ de la política aleatoria convergió en {barridos_iniciales} barridos.")
imprimir_politica(mundo, politica_inicial)

---

## Ejercicio 1: completa iteración de políticas desde cero

En clase construiste `mejora_politica` e `iteracion_politicas` viéndolas ya terminadas. Ahora te toca completarlas tú, sobre el `mundo` de la línea base.

**Pistas:**
- En `mejora_politica_ejercicio`, para cada acción necesitas el estado siguiente (`mundo.estado_siguiente`), la recompensa inmediata (`mundo.recompensa`) y guardar en `valores_q[accion]` el valor de un paso: `r + gamma * V[estado_sig]`. Con `valores_q` ya construido, `max(valores_q, key=valores_q.get)` te da la acción con mayor valor
- En `iteracion_politicas_ejercicio`, el bucle alterna dos llamadas —evaluar con `evaluacion_iterativa_politica` y mejorar con `mejora_politica_ejercicio`— y termina cuando la política nueva es exactamente igual a la anterior (`nueva_politica == politica`)
- Con γ = 0,9 deberías obtener el mismo resultado que viste en clase: tres ciclos y 90 barridos totales, con la política apuntando siempre hacia la meta por el camino más corto

In [ ]:
def mejora_politica_ejercicio(mundo, V, gamma=0.9):
    """Construye una política determinista, codiciosa respecto a V: en cada
    estado, elige la acción que maximiza recompensa + gamma * V(siguiente)."""
    nueva_politica = {}
    for estado in mundo.estados:
        if mundo.es_terminal(estado):
            nueva_politica[estado] = {a: 0.0 for a in mundo.acciones}
            continue

        valores_q = {}
        for accion in mundo.acciones:
            # TODO: calcula el estado siguiente con mundo.estado_siguiente(estado, accion)
            estado_sig = None

            # TODO: calcula la recompensa inmediata con mundo.recompensa(estado, accion, estado_sig)
            r = None

            # TODO: guarda en valores_q[accion] el valor de un paso: r + gamma * V[estado_sig]
            valores_q[accion] = 0

        # TODO: escoge mejor_accion, la clave de valores_q con el valor más alto
        #       (pista: max(diccionario, key=diccionario.get))
        mejor_accion = None

        nueva_politica[estado] = {a: (1.0 if a == mejor_accion else 0.0) for a in mundo.acciones}
    return nueva_politica


def iteracion_politicas_ejercicio(mundo, gamma=0.9, umbral=1e-4):
    """Alterna evaluación completa y mejora hasta que la política deja de cambiar."""
    politica = politica_aleatoria(mundo)
    ciclos = 0
    barridos_totales = 0

    while True:
        ciclos += 1

        # TODO: evalúa la política actual con evaluacion_iterativa_politica(mundo, politica, gamma, umbral)
        V, barridos = None, None
        barridos_totales += barridos

        # TODO: mejora la política con mejora_politica_ejercicio(mundo, V, gamma)
        nueva_politica = None

        # TODO: si nueva_politica es igual a politica, la política ya no cambia: sal del bucle con break
        pass

        politica = nueva_politica

    return politica, V, ciclos, barridos_totales


politica_ej1, V_ej1, ciclos_ej1, barridos_ej1 = iteracion_politicas_ejercicio(mundo, gamma=gamma)

print(f"Ciclos: {ciclos_ej1}, barridos totales: {barridos_ej1}")
imprimir_politica(mundo, politica_ej1)

<details>
<summary>💡 Ver solución</summary>

```python
def mejora_politica_ejercicio(mundo, V, gamma=0.9):
    """Construye una política determinista, codiciosa respecto a V: en cada
    estado, elige la acción que maximiza recompensa + gamma * V(siguiente)."""
    nueva_politica = {}
    for estado in mundo.estados:
        if mundo.es_terminal(estado):
            nueva_politica[estado] = {a: 0.0 for a in mundo.acciones}
            continue

        valores_q = {}
        for accion in mundo.acciones:
            # Estado al que se llega tras aplicar la acción
            estado_sig = mundo.estado_siguiente(estado, accion)

            # Recompensa inmediata de esa transición
            r = mundo.recompensa(estado, accion, estado_sig)

            # Valor de un paso: recompensa inmediata + valor descontado del siguiente estado
            valores_q[accion] = r + gamma * V[estado_sig]

        # La acción codiciosa: la que maximiza el valor de un paso
        mejor_accion = max(valores_q, key=valores_q.get)

        nueva_politica[estado] = {a: (1.0 if a == mejor_accion else 0.0) for a in mundo.acciones}
    return nueva_politica


def iteracion_politicas_ejercicio(mundo, gamma=0.9, umbral=1e-4):
    """Alterna evaluación completa y mejora hasta que la política deja de cambiar."""
    politica = politica_aleatoria(mundo)
    ciclos = 0
    barridos_totales = 0

    while True:
        ciclos += 1

        # Evaluación completa de la política actual
        V, barridos = evaluacion_iterativa_politica(mundo, politica, gamma, umbral)
        barridos_totales += barridos

        # Mejora codiciosa respecto a la V recién calculada
        nueva_politica = mejora_politica_ejercicio(mundo, V, gamma)

        # Si la política ya no cambia, se cumple la ecuación de optimalidad de Bellman
        if nueva_politica == politica:
            break

        politica = nueva_politica

    return politica, V, ciclos, barridos_totales


politica_ej1, V_ej1, ciclos_ej1, barridos_ej1 = iteracion_politicas_ejercicio(mundo, gamma=gamma)

print(f"Ciclos: {ciclos_ej1}, barridos totales: {barridos_ej1}")
imprimir_politica(mundo, politica_ej1)

# Salida esperada con gamma = 0.9:
# Ciclos: 3, barridos totales: 90
#  v | v | v | v
#  v | v | v | v
#  v | v | v | v
#  > | > | > | G
```

**Por qué funciona:** cada ciclo dentro de `while True` hace lo mismo que hiciste a mano en clase, célula a célula: evaluar hasta el final y después mejorar. La condición `nueva_politica == politica` es el criterio de parada de iteración de políticas: cuando la acción codiciosa en cada estado ya coincide con lo que la política hacía, se cumple la ecuación de optimalidad de Bellman y la política es óptima.
</details>

---

## Ejercicio 2: completa iteración de valores y compara sobre un GridWorld con obstáculo

Ahora programas la segunda pieza de hoy: iteración de valores. La diferencia clave respecto a `evaluacion_iterativa_politica` es que, en vez de promediar según una política, cada estado toma el **máximo** entre sus acciones —la ecuación de optimalidad de Bellman—.

Primero construye el GridWorld con un obstáculo, y luego completa el bucle de iteración de valores. Al final, compara el número de iteraciones que tardó en converger frente a `iteracion_politicas_ejercicio` del ejercicio anterior, ejecutada sobre ese mismo GridWorld con obstáculo.

**Pistas:**
- El obstáculo se pasa como `obstaculos=[(1, 1), (2, 1)]`: un muro vertical de dos casillas en medio del tablero. La política óptima tendrá que rodearlo
- Dentro del bucle, para cada estado calculas `valores_q`, una lista con el valor de un paso de cada acción (`r + gamma * V[estado_sig]`), y actualizas `V_nuevo[estado]` con el máximo de esa lista: `max(valores_q)`
- Al terminar de iterar sobre V, deriva la política final con una única llamada a `mejora_politica_ejercicio(mundo, V, gamma)` —una última mirada de un paso, ya no hace falta seguir iterando—
- Compara `ciclos_obs` (de iteración de políticas) y `barridos_obs_pi` (barridos totales) contra `barridos_obs_vi` (barridos de iteración de valores): deberías ver el mismo patrón que en clase, muchos barridos baratos frente a pocos ciclos caros

In [ ]:
# Un obstáculo: un muro vertical de dos casillas en medio del tablero
obstaculos = [(1, 1), (2, 1)]
mundo_obstaculo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0, obstaculos=obstaculos)

print(f"Estados habitables: {len(mundo_obstaculo.estados)} de {4 * 4} casillas totales")


def iteracion_valores_ejercicio(mundo, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Funde evaluación y mejora: cada barrido actualiza V con el máximo
    sobre acciones (ecuación de optimalidad de Bellman), no con un promedio
    según una política fija."""
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valores_q = []
            for accion in mundo.acciones:
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                # TODO: añade a valores_q el valor de un paso de esta acción: r + gamma * V[estado_sig]
                pass

            # TODO: actualiza V_nuevo[estado] con el máximo de valores_q (ecuación de optimalidad de Bellman)
            V_nuevo[estado] = 0.0

            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    # TODO: deriva la política final con una última mirada de un paso:
    #       mejora_politica_ejercicio(mundo, V, gamma)
    politica_final = None

    return politica_final, V, iteracion


# Iteración de políticas sobre el GridWorld con obstáculo (reutilizas el ejercicio 1)
politica_obs_pi, V_obs_pi, ciclos_obs, barridos_obs_pi = iteracion_politicas_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIteración de políticas -> ciclos: {ciclos_obs}, barridos totales: {barridos_obs_pi}")
imprimir_politica(mundo_obstaculo, politica_obs_pi)

# Iteración de valores sobre el mismo GridWorld con obstáculo
politica_obs_vi, V_obs_vi, barridos_obs_vi = iteracion_valores_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIteración de valores -> barridos: {barridos_obs_vi}")
imprimir_politica(mundo_obstaculo, politica_obs_vi)

diferencia_maxima = max(abs(V_obs_pi[s] - V_obs_vi[s]) for s in mundo_obstaculo.estados)
print(f"\nDiferencia máxima entre las dos V*: {diferencia_maxima:.6f}")

<details>
<summary>💡 Ver solución</summary>

```python
obstaculos = [(1, 1), (2, 1)]
mundo_obstaculo = GridWorld(n_filas=4, n_columnas=4, recompensa_paso=-1.0, obstaculos=obstaculos)

print(f"Estados habitables: {len(mundo_obstaculo.estados)} de {4 * 4} casillas totales")


def iteracion_valores_ejercicio(mundo, gamma=0.9, umbral=1e-4, max_iteraciones=1000):
    """Funde evaluación y mejora: cada barrido actualiza V con el máximo
    sobre acciones (ecuación de optimalidad de Bellman), no con un promedio
    según una política fija."""
    V = {estado: 0.0 for estado in mundo.estados}

    for iteracion in range(1, max_iteraciones + 1):
        delta_maximo = 0.0
        V_nuevo = dict(V)

        for estado in mundo.estados:
            if mundo.es_terminal(estado):
                V_nuevo[estado] = 0.0
                continue

            valores_q = []
            for accion in mundo.acciones:
                estado_sig = mundo.estado_siguiente(estado, accion)
                r = mundo.recompensa(estado, accion, estado_sig)
                # Valor de un paso de esta acción
                valores_q.append(r + gamma * V[estado_sig])

            # Ecuación de optimalidad de Bellman: el máximo entre acciones, no un promedio
            V_nuevo[estado] = max(valores_q)

            delta_maximo = max(delta_maximo, abs(V_nuevo[estado] - V[estado]))

        V = V_nuevo
        if delta_maximo < umbral:
            break

    # Una última mirada de un paso para derivar la política óptima
    politica_final = mejora_politica_ejercicio(mundo, V, gamma)

    return politica_final, V, iteracion


politica_obs_pi, V_obs_pi, ciclos_obs, barridos_obs_pi = iteracion_politicas_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIteración de políticas -> ciclos: {ciclos_obs}, barridos totales: {barridos_obs_pi}")
imprimir_politica(mundo_obstaculo, politica_obs_pi)

politica_obs_vi, V_obs_vi, barridos_obs_vi = iteracion_valores_ejercicio(mundo_obstaculo, gamma=gamma)
print(f"\nIteración de valores -> barridos: {barridos_obs_vi}")
imprimir_politica(mundo_obstaculo, politica_obs_vi)

diferencia_maxima = max(abs(V_obs_pi[s] - V_obs_vi[s]) for s in mundo_obstaculo.estados)
print(f"\nDiferencia máxima entre las dos V*: {diferencia_maxima:.6f}")

# Salida esperada con gamma = 0.9:
# Estados habitables: 14 de 16 casillas totales
#
# Iteración de políticas -> ciclos: 3, barridos totales: 91
#  v | > | v | v
#  v | # | v | v
#  v | # | v | v
#  > | > | > | G
#
# Iteración de valores -> barridos: 7
#  v | > | v | v
#  v | # | v | v
#  v | # | v | v
#  > | > | > | G
#
# Diferencia máxima entre las dos V*: 0.000000
```

**Por qué funciona:** el obstáculo reduce los estados habitables de 16 a 14 y obliga a la política a rodearlo —fíjate en la casilla (0, 1), que ahora apunta a la derecha en vez de hacia abajo, precisamente para esquivar el muro—. A pesar de ese cambio en el mapa, el patrón de coste se mantiene idéntico al de la clase: iteración de políticas necesita pocos ciclos (tres) pero cada uno es caro (91 barridos totales), mientras que iteración de valores llega al mismo resultado en muy pocos barridos (siete). Ambos métodos siguen aterrizando en la misma política óptima, porque los dos resuelven exactamente la misma ecuación de optimalidad de Bellman.
</details>

---

## Para seguir practicando

Si te ha sobrado tiempo, vuelve a los dos ejercicios y prueba estas variantes del capítulo:

1. **Cambia el factor de descuento.** Prueba con γ = 0,5 y con γ = 0,99 sobre `mundo_obstaculo`. ¿Cambia el número de barridos que tarda cada método en converger?
2. **Añade un segundo obstáculo o una segunda meta.** Combina `obstaculos` con una lista de dos `estados_terminales` distintos y observa cómo se reparte la cuadrícula entre las dos metas
3. **Agranda el tablero.** Prueba con `n_filas=6, n_columnas=6` y un obstáculo más largo. ¿La diferencia de barridos entre los dos métodos crece o se mantiene igual?

**Siguiente parada:** en el próximo bloque del curso dejas atrás la suposición de conocer el modelo completo del entorno (P y R), y aprendes a evaluar y mejorar políticas solo a partir de la experiencia, con Monte Carlo y diferencias temporales. ¡Guarda este notebook, el `GridWorld` con obstáculos te va a servir de referencia!